In [1]:
import matplotlib.pyplot as plt
import rasterio,numpy as np 


In [3]:
#Here we are collecting the 1st and 99th percentile pixel values for lower and upper bound of our normalization
#for different band in the images (3 for optical) and (2 for sar) we flattened the images to (3.H*W) and (2,H*W) respectively and 
#concatenated different traning images from those we selected few valid set of pixels and from that group of valid pixels we got out 1st and 99th percentile

In [8]:
opt_s,sar_s=[],[]
for i in range(10,56):
    with rasterio.open(f'image_{i}_s2.tif') as f:
        o=f.read().astype(np.float32)
    with rasterio.open(f'image_{i}_s1.tif') as f:
        s=f.read().astype(np.float32)
    o = o.reshape(3,-1)
    s = s.reshape(2,-1)

    ok=np.isfinite(o).all(0) & np.isfinite(s).all(0) & (o.sum(axis=0)>0) # we are selecting all the pixels where each band of optical as well as the sar is finite and optical pixels are not empty
    ok&=o.mean(0)<np.percentile(o.mean(0)[ok],99.5)#dropping the 0.5% brightest pixels outliers
    idx=np.flatnonzero(ok) # returns a 1d array of indices where ok is true
    pick=rng.choice(idx,size=min(200000,len(idx)),replace=False)
    opt_s.append(o[:,pick])
    sar_s.append(s[:,pick])
opt_pixels=np.concatenate(opt_s,axis=1)
sar_pixels=np.concatenate(sar_s,axis=1)
OPT_LO,OPT_HI=np.percentile(opt_pixels,[1,99],axis=1)
SAR_LO,SAR_HI=np.percentile(sar_pixels,[1,99],axis=1)
np.savez("norm_limits.npz",OPT_LO=OPT_LO,OPT_HI=OPT_HI,SAR_LO=SAR_LO,SAR_HI=SAR_HI)


    

In [4]:

data = np.load("norm_limits.npz")
OPT_LO = data["OPT_LO"]
OPT_HI = data["OPT_HI"]
SAR_LO = data["SAR_LO"]
SAR_HI = data["SAR_HI"]

# Print shapes to verify
print("OPT_LO shape:", OPT_LO.shape)
print("OPT_HI shape:", OPT_HI.shape)
print("SAR_LO shape:", SAR_LO.shape)
print("SAR_HI shape:", SAR_HI.shape)

# Print values
print("\nOptical Lower Bounds:\n", OPT_LO)
print("Optical Upper Bounds:\n", OPT_HI)
print("SAR Lower Bounds:\n", SAR_LO)
print("SAR Upper Bounds:\n", SAR_HI)

OPT_LO shape: (3,)
OPT_HI shape: (3,)
SAR_LO shape: (2,)
SAR_HI shape: (2,)

Optical Lower Bounds:
 [129. 211. 127.]
Optical Upper Bounds:
 [3948. 3920. 4007.]
SAR Lower Bounds:
 [-23.20216179 -36.91700001]
SAR Upper Bounds:
 [ 0.98891147 -8.63401318]


In [2]:
import sys, numpy as np, rasterio, torch
import torch.nn.functional as F


def make_cloud_weight(g):
    up = lambda t: F.interpolate(t, size=(256, 256), mode='bicubic', align_corners=False)
    cover = 0.1 + 0.6 * torch.rand(1, generator=g).item()
    n = 0.7 * up(torch.rand(1, 1, 8, 8, generator=g)) + 0.3 * up(torch.rand(1, 1, 32, 32, generator=g))
    thr = torch.quantile(n, 1 - cover)
    m = torch.sigmoid((n - thr) / 0.03)[0]                    
    alpha = 0.5 + 0.5 * torch.rand(1, generator=g).item()
    return (alpha * m).clamp(0, 1)

In [3]:
                         
def process_pair(opt_path, sar_path, out_path, limits="norm_limits.npz",
                 stride=128, size=256, seed=0):
    lim = np.load(limits)
    OPT_LO, OPT_HI = lim["OPT_LO"].reshape(3, 1, 1), lim["OPT_HI"].reshape(3, 1, 1)
    SAR_LO, SAR_HI = lim["SAR_LO"].reshape(2, 1, 1), lim["SAR_HI"].reshape(2, 1, 1)

    with rasterio.open(opt_path) as f: opt = f.read().astype(np.float32)   # [3,H,W]
    with rasterio.open(sar_path) as f: sar = f.read().astype(np.float32)   # [2,H,W]
    H, W = min(opt.shape[1], sar.shape[1]), min(opt.shape[2], sar.shape[2])
    opt, sar = opt[:, :H, :W], sar[:, :H, :W]          # harmless if sizes already match

    # valid mask on RAW data (before nan_to_num hides the missing pixels)
    valid = np.isfinite(opt).all(0) & np.isfinite(sar).all(0) & (opt.sum(0) > 0)

    # normalize with the fixed limits
    opt = np.clip((opt - OPT_LO) / (OPT_HI - OPT_LO), 0, 1)
    sar = (np.clip(sar, SAR_LO, SAR_HI) - SAR_LO) / (SAR_HI - SAR_LO) * 2 - 1
    opt = np.nan_to_num(opt, nan=0.0)
    sar = np.nan_to_num(sar, nan=-1.0)

    # cut patches, add cloud weight, stack
    out, k = [], 0
    for r in range(0, H - size + 1, stride):
        for c in range(0, W - size + 1, stride):
            o = opt[:, r:r+size, c:c+size]
            g = torch.Generator(); g.manual_seed(seed * 1_000_003 + k); k += 1
            w = make_cloud_weight(g).numpy()                            # [1,256,256]
            s = sar[:, r:r+size, c:c+size]
            out.append(np.concatenate([o, s, w], axis=0))               # [6,256,256]

    data = np.stack(out).astype(np.float16)                             # [N,6,256,256]
    np.save(out_path, data)
    print(out_path, data.shape, "| opt", data[:, 0:3].min(), data[:, 0:3].max(),
          "| sar", data[:, 3:5].min(), data[:, 3:5].max(),
          "| mean cloud weight", float(data[:, 5].mean()))



In [287]:
for i in range (10,56):
     process_pair(f'image_{i}_s2.tif',f'image_{i}_s1.tif',f'image_{i}', limits="norm_limits.npz",
                 stride=128, size=256, seed=i//10)

    

image_10 (256, 6, 256, 256) | opt 0.004974 1.0 | sar -1.0 1.0 | mean cloud weight 0.284423828125
image_11 (256, 6, 256, 256) | opt 0.0 1.0 | sar -1.0 1.0 | mean cloud weight 0.284423828125
image_12 (256, 6, 256, 256) | opt 0.02422 1.0 | sar -1.0 1.0 | mean cloud weight 0.284423828125
image_13 (256, 6, 256, 256) | opt 0.004852 1.0 | sar -1.0 1.0 | mean cloud weight 0.284423828125
image_14 (256, 6, 256, 256) | opt 0.0 1.0 | sar -1.0 1.0 | mean cloud weight 0.284423828125
image_15 (256, 6, 256, 256) | opt 0.001887 1.0 | sar -1.0 1.0 | mean cloud weight 0.284423828125
image_16 (256, 6, 256, 256) | opt 0.0 1.0 | sar -1.0 1.0 | mean cloud weight 0.284423828125
image_17 (256, 6, 256, 256) | opt 0.01482 1.0 | sar -1.0 1.0 | mean cloud weight 0.284423828125
image_18 (256, 6, 256, 256) | opt 0.05826 1.0 | sar -1.0 1.0 | mean cloud weight 0.284423828125
image_19 (256, 6, 256, 256) | opt 0.02087 1.0 | sar -1.0 1.0 | mean cloud weight 0.284423828125
image_20 (256, 6, 256, 256) | opt 0.0 1.0 | sar -